In [0]:
from pyspark.sql.functions import *

In [0]:
# ============================================================
# STEP 1 : Read Bronze Products Table
# ============================================================

products = spark.table("workspace.default.bronze_products")

print("Bronze Products Table Loaded Successfully")

display(products)

Bronze Products Table Loaded Successfully


product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
PROD_000001,music,66,512,5,18964,11,25,52
PROD_000002,food,40,452,4,25798,57,42,49
PROD_000003,office,69,121,5,24989,45,16,10
PROD_000004,music,46,717,4,18661,91,8,48
PROD_000005,toys,29,632,5,26347,79,32,11
PROD_000006,books,24,845,1,12830,52,31,68
PROD_000007,sports,34,563,3,12265,66,7,20
PROD_000008,books,40,247,4,13678,21,19,63
PROD_000009,garden,55,170,3,19359,80,8,27
PROD_000010,toys,34,750,1,16878,45,17,71


In [0]:
# ============================================================
# STEP 2 : Remove Duplicate Records
# ============================================================

products = products.dropDuplicates(["product_id"])

print("Duplicate Products Removed Successfully")

Duplicate Products Removed Successfully


In [0]:
# ============================================================
# STEP 3 : Remove Leading and Trailing Spaces
# ============================================================

string_columns = [c for c, t in products.dtypes if t == "string"]

for column in string_columns:
    products = products.withColumn(column, trim(col(column)))

print("Whitespace Removed Successfully")

Whitespace Removed Successfully


In [0]:
# ============================================================
# STEP 4 : Handle Missing Category
# ============================================================

products = products.fillna({
    "product_category_name": "Unknown"
})

print("Missing Categories Handled")

Missing Categories Handled


In [0]:
# ============================================================
# STEP 5 : Standardize Category Names
# ============================================================

products = products.withColumn(
    "product_category_name",
    initcap(col("product_category_name"))
)

print("Product Categories Standardized")

Product Categories Standardized


In [0]:
# ============================================================
# STEP 6 : Fill Missing Numeric Values
# ============================================================

products = products.fillna({
    "product_weight_g": 0,
    "product_length_cm": 0,
    "product_height_cm": 0,
    "product_width_cm": 0
})

print("Missing Numeric Values Filled")

Missing Numeric Values Filled


In [0]:
# ============================================================
# STEP 7 : Create Product Volume
# ============================================================

products = products.withColumn(
    "product_volume_cm3",
    col("product_length_cm") *
    col("product_height_cm") *
    col("product_width_cm")
)

print("Product Volume Created")

Product Volume Created


In [0]:
# ============================================================
# STEP 8 : Create Size Category
# ============================================================

products = products.withColumn(
    "size_category",
    when(col("product_volume_cm3") < 1000, "Small")
    .when(col("product_volume_cm3") < 10000, "Medium")
    .otherwise("Large")
)

print("Size Category Created")

Size Category Created


In [0]:
# ============================================================
# STEP 9 : Create Weight Category
# ============================================================

products = products.withColumn(
    "weight_category",
    when(col("product_weight_g") < 500, "Light")
    .when(col("product_weight_g") < 2000, "Medium")
    .otherwise("Heavy")
)

print("Weight Category Created")

Weight Category Created


In [0]:
# ============================================================
# STEP 10 : Create Shipping Type
# ============================================================

products = products.withColumn(
    "shipping_type",
    when(col("product_weight_g") >= 5000, "Special Handling")
    .otherwise("Standard")
)

print("Shipping Type Created")

Shipping Type Created


In [0]:
# ============================================================
# STEP 11 : Validate Product Data
# ============================================================

products = products.filter(
    (col("product_weight_g") >= 0) &
    (col("product_length_cm") >= 0) &
    (col("product_height_cm") >= 0) &
    (col("product_width_cm") >= 0)
)

print("Product Validation Completed")

Product Validation Completed


In [0]:
# ============================================================
# STEP 12 : Preview Silver Products
# ============================================================

display(products)

product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_volume_cm3,size_category,weight_category,shipping_type
PROD_000001,Music,66,512,5,18964,11,25,52,14300,Large,Heavy,Special Handling
PROD_000002,Food,40,452,4,25798,57,42,49,117306,Large,Heavy,Special Handling
PROD_000003,Office,69,121,5,24989,45,16,10,7200,Medium,Heavy,Special Handling
PROD_000004,Music,46,717,4,18661,91,8,48,34944,Large,Heavy,Special Handling
PROD_000005,Toys,29,632,5,26347,79,32,11,27808,Large,Heavy,Special Handling
PROD_000006,Books,24,845,1,12830,52,31,68,109616,Large,Heavy,Special Handling
PROD_000007,Sports,34,563,3,12265,66,7,20,9240,Medium,Heavy,Special Handling
PROD_000008,Books,40,247,4,13678,21,19,63,25137,Large,Heavy,Special Handling
PROD_000009,Garden,55,170,3,19359,80,8,27,17280,Large,Heavy,Special Handling
PROD_000010,Toys,34,750,1,16878,45,17,71,54315,Large,Heavy,Special Handling


In [0]:
# ============================================================
# STEP 13 : Save Silver Products Table
# ============================================================

products.write \
.mode("overwrite") \
.saveAsTable("workspace.default.silver_products")

print("Silver Products Table Created Successfully")

Silver Products Table Created Successfully


In [0]:
# ============================================================
# STEP 14 : Verify Silver Products
# ============================================================

silver_products = spark.table("workspace.default.silver_products")

display(silver_products)

silver_products.printSchema()

print("Total Products :", silver_products.count())

product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_volume_cm3,size_category,weight_category,shipping_type
PROD_000005,Toys,29,632,5,26347,79,32,11,27808,Large,Heavy,Special Handling
PROD_000008,Books,40,247,4,13678,21,19,63,25137,Large,Heavy,Special Handling
PROD_000010,Toys,34,750,1,16878,45,17,71,54315,Large,Heavy,Special Handling
PROD_000013,Health,20,644,3,22228,33,20,28,18480,Large,Heavy,Special Handling
PROD_000016,Toys,35,994,4,19772,23,25,78,44850,Large,Heavy,Special Handling
PROD_000031,Sports,43,733,2,16136,87,36,20,62640,Large,Heavy,Special Handling
PROD_000036,Beauty,70,898,3,23920,37,13,39,18759,Large,Heavy,Special Handling
PROD_000042,Garden,50,763,4,2292,26,6,30,4680,Medium,Heavy,Standard
PROD_000043,Food,25,941,6,21333,51,35,80,142800,Large,Heavy,Special Handling
PROD_000063,Furniture,52,400,5,14054,30,15,45,20250,Large,Heavy,Special Handling


root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_length: long (nullable = true)
 |-- product_description_length: long (nullable = true)
 |-- product_photos_qty: long (nullable = true)
 |-- product_weight_g: long (nullable = true)
 |-- product_length_cm: long (nullable = true)
 |-- product_height_cm: long (nullable = true)
 |-- product_width_cm: long (nullable = true)
 |-- product_volume_cm3: long (nullable = true)
 |-- size_category: string (nullable = true)
 |-- weight_category: string (nullable = true)
 |-- shipping_type: string (nullable = true)

Total Products : 3000
